# Modern Marketing Email Campaign Optimization with XGBoost

This notebook modernizes the original email-campaign project while keeping **XGBoost** as the machine-learning model.

## Business goals

1. Predict the probability that an eligible user clicks the link in a marketing email.
2. Rank users so a limited send budget can be allocated efficiently.
3. Compare model-based targeting with random targeting at the **same email volume**.
4. Identify descriptive patterns and create product/marketing hypotheses.
5. Save a reproducible model, metadata, category schema, and scored records.

## Main upgrades

- Vectorized label creation with `Series.isin`.
- Explicit schema, ID, range, duplicate, and funnel checks.
- A pre-send feature contract that excludes post-send outcomes such as `opened`.
- Stratified training, validation, and test sets.
- Native XGBoost categorical features with histogram trees.
- Regularization and validation-based early stopping.
- PR-AUC, ROC-AUC, log loss, Brier score, calibration, lift, precision, recall, and F1.
- Threshold selection on validation data only.
- Top-k campaign-policy analysis: email savings, click capture, CTR lift, and lift over random.
- Clear separation between predictive association and causal impact.

> Run the notebook from top to bottom using the real CSV files. Final results depend on the real data.

## 0. Environment

Uncomment the package-installation cell when needed, then restart the kernel.

In [ ]:
# %pip install -q "pandas>=2.2" "numpy>=2.0" "scikit-learn>=1.5" \
#     "xgboost>=3.1,<4" "matplotlib>=3.8" "seaborn>=0.13" "scipy>=1.13"

In [ ]:
from __future__ import annotations

import json
import os
import platform
import sys
import warnings
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
import xgboost as xgb
from scipy.stats import norm
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    classification_report,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
sns.set_theme(style="whitegrid", context="notebook")

print(f"Python:        {sys.version.split()[0]}")
print(f"pandas:        {pd.__version__}")
print(f"NumPy:         {np.__version__}")
print(f"scikit-learn:  {sklearn.__version__}")
print(f"XGBoost:       {xgb.__version__}")
print(f"Platform:      {platform.platform()}")

## 1. Configuration and prediction-time contract

The primary score is generated **before an email is sent**.

Allowed predictors:

- email text type,
- personalized versus generic version,
- planned send hour,
- planned weekday,
- user country,
- past purchase count.

Excluded variables:

- `opened`: an outcome that happens after the send;
- `clicked`: the target itself;
- `email_id`: an identifier, not a stable behavior feature.

This prevents prediction-time leakage.

In [ ]:
BASE_DIR = Path.cwd()

EMAIL_TABLE_PATH = Path(os.getenv("EMAIL_TABLE_PATH", BASE_DIR / "email_table.csv"))
OPENED_TABLE_PATH = Path(os.getenv("OPENED_TABLE_PATH", BASE_DIR / "email_opened_table.csv"))
CLICKED_TABLE_PATH = Path(os.getenv("CLICKED_TABLE_PATH", BASE_DIR / "link_clicked_table.csv"))

TARGET = "clicked"
OPEN_TARGET = "opened"

CATEGORICAL_FEATURES = ["email_text", "email_version", "weekday", "user_country"]
BASE_NUMERIC_FEATURES = ["hour", "user_past_purchases"]
ENGINEERED_NUMERIC_FEATURES = ["hour_sin", "hour_cos", "is_weekend"]
FEATURES = CATEGORICAL_FEATURES + BASE_NUMERIC_FEATURES + ENGINEERED_NUMERIC_FEATURES

RUN_TUNING = False
TUNING_ITERATIONS = 16

# The binary threshold is selected on validation data.
THRESHOLD_OBJECTIVE = "f1"

# The campaign policy is evaluated at fixed fractions of the eligible population.
TARGET_FRACTIONS = [0.05, 0.10, 0.20, 0.30, 0.50, 1.00]
DEFAULT_TARGET_FRACTION = 0.30

# Optional economics. Leave as None to select the threshold by F1.
VALUE_PER_CLICK: float | None = None
COST_PER_EMAIL: float | None = None

OUTPUT_DIR = Path("artifacts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [EMAIL_TABLE_PATH, OPENED_TABLE_PATH, CLICKED_TABLE_PATH]:
    print(path)

## 2. Load and validate source tables

Expected files:

- `email_table.csv`: one row per sent email.
- `email_opened_table.csv`: IDs of opened emails.
- `link_clicked_table.csv`: IDs of emails whose link was clicked.

In [ ]:
def require_file(path: Path) -> None:
    if not path.exists():
        raise FileNotFoundError(
            f"Could not find {path}. Put the file beside the notebook or set its environment variable."
        )

for source_path in [EMAIL_TABLE_PATH, OPENED_TABLE_PATH, CLICKED_TABLE_PATH]:
    require_file(source_path)

email_table = pd.read_csv(EMAIL_TABLE_PATH)
opened_table = pd.read_csv(OPENED_TABLE_PATH)
clicked_table = pd.read_csv(CLICKED_TABLE_PATH)

required_email_columns = {
    "email_id",
    "email_text",
    "email_version",
    "hour",
    "weekday",
    "user_country",
    "user_past_purchases",
}
required_event_columns = {"email_id"}

missing_email = sorted(required_email_columns - set(email_table.columns))
missing_opened = sorted(required_event_columns - set(opened_table.columns))
missing_clicked = sorted(required_event_columns - set(clicked_table.columns))

if missing_email:
    raise ValueError(f"email_table.csv is missing: {missing_email}")
if missing_opened:
    raise ValueError(f"email_opened_table.csv is missing: {missing_opened}")
if missing_clicked:
    raise ValueError(f"link_clicked_table.csv is missing: {missing_clicked}")

print(f"email_table:   {email_table.shape}")
print(f"opened_table:  {opened_table.shape}")
print(f"clicked_table: {clicked_table.shape}")
display(email_table.head())

## 3. Data quality and vectorized label construction

The older workflow created labels with a Python function applied to every row and repeatedly scanned ID lists. This version uses vectorized membership checks.

The email table must contain one row per `email_id`. Event-table duplicates are harmless after conversion to unique ID sets, but they are reported.

In [ ]:
def data_quality_report(frame: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame(
        {
            "dtype": frame.dtypes.astype(str),
            "missing": frame.isna().sum(),
            "missing_pct": frame.isna().mean(),
            "unique": frame.nunique(dropna=False),
        }
    ).sort_index()

display(data_quality_report(email_table))

print("Exact duplicate rows")
print(f"email table:   {email_table.duplicated().sum():,}")
print(f"opened table:  {opened_table.duplicated().sum():,}")
print(f"clicked table: {clicked_table.duplicated().sum():,}")

duplicate_email_ids = int(email_table["email_id"].duplicated().sum())
if duplicate_email_ids:
    raise ValueError(
        f"email_table must have one row per email_id; found {duplicate_email_ids:,} duplicate IDs."
    )

In [ ]:
df = email_table.copy()

for column in CATEGORICAL_FEATURES:
    df[column] = (
        df[column]
        .astype("string")
        .str.strip()
        .str.lower()
        .fillna("__missing__")
    )

for column in ["email_id", "hour", "user_past_purchases"]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

if df["email_id"].isna().any():
    raise ValueError("email_table contains missing or invalid email_id values.")

opened_ids = set(
    pd.to_numeric(opened_table["email_id"], errors="coerce").dropna().unique()
)
clicked_ids = set(
    pd.to_numeric(clicked_table["email_id"], errors="coerce").dropna().unique()
)
sent_ids = set(df["email_id"].unique())

unknown_opened = opened_ids - sent_ids
unknown_clicked = clicked_ids - sent_ids

if unknown_opened:
    print(f"Warning: {len(unknown_opened):,} opened IDs are absent from email_table.")
if unknown_clicked:
    print(f"Warning: {len(unknown_clicked):,} clicked IDs are absent from email_table.")

df[OPEN_TARGET] = df["email_id"].isin(opened_ids).astype("int8")
df[TARGET] = df["email_id"].isin(clicked_ids).astype("int8")

clicked_without_open = int(((df[TARGET] == 1) & (df[OPEN_TARGET] == 0)).sum())
if clicked_without_open:
    print(
        f"Warning: {clicked_without_open:,} clicked records are not marked opened. "
        "Review event attribution."
    )

display(df.head())

## 4. Explicit cleaning rules

- `hour` must be from 1 to 24, matching the original data convention.
- `user_past_purchases` must be non-negative.
- weekdays are normalized to lowercase names.
- invalid predictor rows are reported and removed.
- event labels are never imputed.

In [ ]:
VALID_WEEKDAYS = [
    "monday", "tuesday", "wednesday", "thursday",
    "friday", "saturday", "sunday",
]

invalid_hour = ~df["hour"].between(1, 24, inclusive="both")
invalid_purchases = df["user_past_purchases"].lt(0)
invalid_weekday = ~df["weekday"].isin(VALID_WEEKDAYS)

quality_issues = pd.Series(
    {
        "invalid_or_missing_hour": int((invalid_hour | df["hour"].isna()).sum()),
        "negative_or_missing_purchases": int(
            (invalid_purchases | df["user_past_purchases"].isna()).sum()
        ),
        "unexpected_weekday": int(invalid_weekday.sum()),
    },
    name="rows",
)
display(quality_issues.to_frame())

valid_rows = (
    df["hour"].between(1, 24, inclusive="both")
    & df["user_past_purchases"].ge(0)
    & df["weekday"].isin(VALID_WEEKDAYS)
)

rows_before = len(df)
df = df.loc[valid_rows].copy()
print(f"Rows retained: {len(df):,} / {rows_before:,}")

## 5. Funnel KPIs and confidence intervals

\[
\text{Open rate}=\frac{\text{opens}}{\text{sent}}
\]

\[
\text{Click-through rate (CTR)}=\frac{\text{clicks}}{\text{sent}}
\]

\[
\text{Click-to-open rate (CTOR)}=\frac{\text{clicks}}{\text{opens}}
\]

Wilson intervals are used for segment CTRs so very small groups are easier to recognize as uncertain.

In [ ]:
def wilson_interval(successes, totals, z: float = 1.96):
    successes = np.asarray(successes, dtype=float)
    totals = np.asarray(totals, dtype=float)
    with np.errstate(divide="ignore", invalid="ignore"):
        rate = successes / totals
        denominator = 1 + z**2 / totals
        center = (rate + z**2 / (2 * totals)) / denominator
        margin = (
            z
            * np.sqrt(rate * (1 - rate) / totals + z**2 / (4 * totals**2))
            / denominator
        )
    lower = np.where(totals > 0, center - margin, np.nan)
    upper = np.where(totals > 0, center + margin, np.nan)
    return lower, upper

sent_count = len(df)
open_count = int(df[OPEN_TARGET].sum())
click_count = int(df[TARGET].sum())

open_rate = open_count / sent_count
ctr = click_count / sent_count
ctor = click_count / open_count if open_count else np.nan

funnel = pd.DataFrame(
    {
        "stage": ["sent", "opened", "clicked"],
        "count": [sent_count, open_count, click_count],
        "share_of_sent": [1.0, open_rate, ctr],
    }
)

display(funnel)
print(f"Open rate:          {open_rate:.3%}")
print(f"CTR:                {ctr:.3%}")
print(f"Click-to-open rate: {ctor:.3%}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(data=funnel, x="stage", y="count", color="#4C72B0", ax=ax)
ax.set_title("Marketing Email Funnel")
ax.set_xlabel("")
ax.set_ylabel("Records")
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f")
plt.tight_layout()
plt.show()

## 6. Segment analysis

Each table includes send volume, opens, clicks, open rate, CTR, CTOR, lift versus overall CTR, confidence intervals, and a volume-aware click gap.

In [ ]:
def segment_performance(frame: pd.DataFrame, column: str) -> pd.DataFrame:
    result = (
        frame.groupby(column, observed=True, dropna=False)
        .agg(
            sent=("email_id", "size"),
            opened=(OPEN_TARGET, "sum"),
            clicked=(TARGET, "sum"),
        )
        .reset_index()
    )
    result["share_of_sends"] = result["sent"] / len(frame)
    result["open_rate"] = result["opened"] / result["sent"]
    result["ctr"] = result["clicked"] / result["sent"]
    result["ctor"] = result["clicked"] / result["opened"].replace(0, np.nan)
    result["ctr_lift_vs_overall"] = result["ctr"] / frame[TARGET].mean()
    low, high = wilson_interval(result["clicked"], result["sent"])
    result["ctr_ci_low"] = low
    result["ctr_ci_high"] = high
    result["click_gap_vs_overall_rate"] = (
        result["clicked"] - result["sent"] * frame[TARGET].mean()
    )
    return result.sort_values(["ctr", "sent"], ascending=[False, False]).reset_index(drop=True)

for feature in ["email_text", "email_version", "user_country", "weekday"]:
    print(f"\nPerformance by {feature}")
    display(segment_performance(df, feature))

In [ ]:
def plot_segment_ctr(frame: pd.DataFrame, column: str, order=None) -> None:
    summary = segment_performance(frame, column)
    if order is not None:
        summary[column] = pd.Categorical(summary[column], categories=order, ordered=True)
        summary = summary.sort_values(column)

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    sns.barplot(data=summary, x=column, y="sent", color="#4C72B0", ax=axes[0])
    axes[0].set_title(f"Send Volume by {column}")
    axes[0].tick_params(axis="x", rotation=35)

    sns.barplot(data=summary, x=column, y="ctr", color="#55A868", ax=axes[1])
    axes[1].axhline(frame[TARGET].mean(), linestyle="--", label="Overall CTR")
    axes[1].set_title(f"CTR by {column}")
    axes[1].tick_params(axis="x", rotation=35)
    axes[1].legend()
    plt.tight_layout()
    plt.show()

plot_segment_ctr(df, "email_text")
plot_segment_ctr(df, "email_version")
plot_segment_ctr(df, "user_country")
plot_segment_ctr(df, "weekday", order=VALID_WEEKDAYS)

In [ ]:
hour_performance = segment_performance(df, "hour").sort_values("hour")
display(hour_performance)

fig, ax = plt.subplots(figsize=(11, 4.5))
sns.lineplot(data=hour_performance, x="hour", y="ctr", marker="o", color="#4C72B0", ax=ax)
ax.axhline(ctr, linestyle="--", label="Overall CTR")
ax.set_title("CTR by Planned Send Hour")
ax.set_ylabel("CTR")
ax.legend()
plt.tight_layout()
plt.show()

purchase_bins = [-0.1, 0, 1, 2, 3, 5, 7, 10, np.inf]
purchase_labels = ["0", "1", "2", "3", "4-5", "6-7", "8-10", "11+"]
df["past_purchase_band"] = pd.cut(
    df["user_past_purchases"],
    bins=purchase_bins,
    labels=purchase_labels,
)

purchase_performance = segment_performance(df, "past_purchase_band")
display(purchase_performance)

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(
    data=purchase_performance,
    x="past_purchase_band",
    y="ctr",
    order=purchase_labels,
    color="#4C72B0",
    ax=ax,
)
ax.axhline(ctr, linestyle="--", label="Overall CTR")
ax.set_title("CTR by Past-Purchase Band")
ax.set_xlabel("Past purchases")
ax.set_ylabel("CTR")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Feature engineering

Hours are cyclical: hour 24 and hour 1 are neighbors. Sine/cosine encoding preserves this relationship:

\[
\text{hour\_sin}=\sin\left(2\pi\frac{\text{hour}-1}{24}\right)
\]

\[
\text{hour\_cos}=\cos\left(2\pi\frac{\text{hour}-1}{24}\right)
\]

The raw hour is retained because trees can also learn useful intervals. A weekend flag complements the full weekday category.

In [ ]:
def build_model_features(frame: pd.DataFrame) -> pd.DataFrame:
    features = frame[CATEGORICAL_FEATURES + BASE_NUMERIC_FEATURES].copy()

    for column in CATEGORICAL_FEATURES:
        features[column] = (
            features[column]
            .astype("string")
            .str.strip()
            .str.lower()
            .fillna("__missing__")
        )

    for column in BASE_NUMERIC_FEATURES:
        features[column] = pd.to_numeric(features[column], errors="coerce")

    angle = 2 * np.pi * (features["hour"] - 1) / 24
    features["hour_sin"] = np.sin(angle)
    features["hour_cos"] = np.cos(angle)
    features["is_weekend"] = features["weekday"].isin(["saturday", "sunday"]).astype("int8")

    return features[FEATURES]

X = build_model_features(df)
y = df[TARGET].astype("int8")

display(X.head())
print(f"Feature matrix: {X.shape}")
print(f"Click rate:     {y.mean():.3%}")

## 8. Leakage-safe train, validation, and test sets

- Training: 70% — learns model parameters.
- Validation: 15% — early stopping and threshold/policy selection.
- Test: 15% — final evaluation only.

The split is stratified because clicks are rare. If dates become available, replace this with a chronological split for production realism.

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE,
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE,
)

split_summary = pd.DataFrame(
    {
        "rows": [len(X_train), len(X_valid), len(X_test)],
        "clicks": [int(y_train.sum()), int(y_valid.sum()), int(y_test.sum())],
        "click_rate": [y_train.mean(), y_valid.mean(), y_test.mean()],
    },
    index=["train", "validation", "test"],
)
display(split_summary)

## 9. Native categorical features

Category levels are learned from the training split only. The explicit `__unknown__` value safely handles future categories absent from training.

In [ ]:
def learn_category_schema(training_frame, categorical_columns):
    schema = {}
    for column in categorical_columns:
        levels = sorted(
            training_frame[column].astype("string").fillna("__missing__").unique().tolist()
        )
        for special_value in ["__missing__", "__unknown__"]:
            if special_value not in levels:
                levels.append(special_value)
        schema[column] = levels
    return schema

def apply_category_schema(frame, schema):
    transformed = frame.copy()
    for column, levels in schema.items():
        values = transformed[column].astype("string").fillna("__missing__")
        values = values.where(values.isin(levels), "__unknown__")
        transformed[column] = pd.Categorical(values, categories=levels)
    return transformed

CATEGORY_SCHEMA = learn_category_schema(X_train, CATEGORICAL_FEATURES)

X_train_cat = apply_category_schema(X_train, CATEGORY_SCHEMA)
X_valid_cat = apply_category_schema(X_valid, CATEGORY_SCHEMA)
X_test_cat = apply_category_schema(X_test, CATEGORY_SCHEMA)

print(X_train_cat.dtypes)

## 10. Prior-probability baseline

The dummy model predicts the training click rate for every user. XGBoost should beat it on probability quality and top-ranked click concentration.

In [ ]:
dummy_model = DummyClassifier(strategy="prior", random_state=RANDOM_STATE)
dummy_model.fit(np.zeros((len(X_train_cat), 1)), y_train)
dummy_test_probability = dummy_model.predict_proba(
    np.zeros((len(X_test_cat), 1))
)[:, 1]
print(f"Dummy probability: {dummy_test_probability[0]:.6f}")

## 11. Optional randomized hyperparameter search

Set `RUN_TUNING=True` to perform stratified randomized search using average precision. The final number of trees is still selected separately by early stopping.

In [ ]:
DEFAULT_PARAMS: dict[str, Any] = {
    "learning_rate": 0.04,
    "max_depth": 4,
    "min_child_weight": 5,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.10,
    "reg_lambda": 3.0,
    "gamma": 0.0,
    "max_delta_step": 1,
    "max_cat_to_onehot": 4,
}

selected_params = DEFAULT_PARAMS.copy()

if RUN_TUNING:
    tuning_model = xgb.XGBClassifier(
        objective="binary:logistic",
        tree_method="hist",
        enable_categorical=True,
        n_estimators=800,
        eval_metric="aucpr",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

    parameter_distributions = {
        "learning_rate": [0.02, 0.04, 0.06, 0.08],
        "max_depth": [3, 4, 5, 6],
        "min_child_weight": [1, 3, 5, 8, 12],
        "subsample": [0.70, 0.80, 0.90, 1.00],
        "colsample_bytree": [0.70, 0.80, 0.90, 1.00],
        "reg_alpha": [0.0, 0.05, 0.10, 0.50, 1.0],
        "reg_lambda": [1.0, 2.0, 3.0, 5.0, 10.0],
        "gamma": [0.0, 0.05, 0.10, 0.25],
        "max_delta_step": [0, 1, 2],
    }

    cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
    search = RandomizedSearchCV(
        tuning_model,
        parameter_distributions,
        n_iter=TUNING_ITERATIONS,
        scoring="average_precision",
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=1,
        verbose=1,
        refit=False,
    )
    search.fit(X_train_cat, y_train)
    selected_params.update(search.best_params_)
    print(f"Best CV average precision: {search.best_score_:.6f}")
else:
    print("Tuning skipped; using robust default parameters.")

display(pd.Series(selected_params).to_frame("value"))

## 12. XGBoost training with early stopping

A large maximum of 5,000 trees is safe because validation log loss controls early stopping. PR-AUC is tracked at the same time.

SMOTE is not used. Preserving the original prevalence makes probabilities easier to interpret. Any future class-weighting experiment should be compared on both ranking and calibration.

In [ ]:
early_stopping = xgb.callback.EarlyStopping(
    rounds=100,
    metric_name="logloss",
    data_name="validation_0",
    maximize=False,
    save_best=True,
)

model = xgb.XGBClassifier(
    objective="binary:logistic",
    tree_method="hist",
    enable_categorical=True,
    n_estimators=5_000,
    eval_metric=["aucpr", "logloss"],
    callbacks=[early_stopping],
    random_state=RANDOM_STATE,
    n_jobs=-1,
    **selected_params,
)

model.fit(
    X_train_cat,
    y_train,
    eval_set=[(X_valid_cat, y_valid)],
    verbose=False,
)

print(f"Best iteration: {model.best_iteration:,}")
print(f"Best validation score: {model.best_score:.6f}")

In [ ]:
history = model.evals_result()["validation_0"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(history["logloss"])
axes[0].axvline(model.best_iteration, linestyle="--", label="Best iteration")
axes[0].set_title("Validation Log Loss")
axes[0].set_xlabel("Boosting round")
axes[0].legend()

axes[1].plot(history["aucpr"])
axes[1].axvline(model.best_iteration, linestyle="--", label="Best iteration")
axes[1].set_title("Validation PR-AUC")
axes[1].set_xlabel("Boosting round")
axes[1].legend()

plt.tight_layout()
plt.show()

## 13. Select a binary threshold on validation data

The default threshold maximizes validation F1. In production, replace this with expected profit, a capacity constraint, or a minimum precision/recall rule.

The test set is not used to choose the threshold.

In [ ]:
valid_probability = model.predict_proba(X_valid_cat)[:, 1]
precision_values, recall_values, thresholds = precision_recall_curve(
    y_valid, valid_probability
)

precision_at_threshold = precision_values[:-1]
recall_at_threshold = recall_values[:-1]
f1_values = (
    2 * precision_at_threshold * recall_at_threshold
    / (precision_at_threshold + recall_at_threshold + 1e-12)
)

threshold_table = pd.DataFrame(
    {
        "threshold": thresholds,
        "precision": precision_at_threshold,
        "recall": recall_at_threshold,
        "f1": f1_values,
    }
)

best_f1_row = threshold_table.loc[threshold_table["f1"].idxmax()]
selected_threshold = float(best_f1_row["threshold"])
threshold_reason = "maximum validation F1"

if VALUE_PER_CLICK is not None and COST_PER_EMAIL is not None:
    net_values = []
    y_valid_array = y_valid.to_numpy()
    for threshold in threshold_table["threshold"]:
        decisions = valid_probability >= threshold
        true_positives = int(((decisions == 1) & (y_valid_array == 1)).sum())
        emails_sent = int(decisions.sum())
        net_values.append(
            true_positives * VALUE_PER_CLICK - emails_sent * COST_PER_EMAIL
        )
    threshold_table["net_value"] = net_values
    best_value_row = threshold_table.loc[threshold_table["net_value"].idxmax()]
    selected_threshold = float(best_value_row["threshold"])
    threshold_reason = "maximum validation net value"

print(f"Selected threshold: {selected_threshold:.6f} ({threshold_reason})")
display(threshold_table.sort_values("f1", ascending=False).head(10))

## 14. Final test evaluation

Average precision is especially important because clicks are rare. ROC-AUC, log loss, Brier score, calibration, and threshold metrics answer complementary questions.

In [ ]:
def evaluate_probabilities(y_true, probability, threshold):
    y_array = np.asarray(y_true)
    predictions = (probability >= threshold).astype("int8")
    return pd.Series(
        {
            "roc_auc": roc_auc_score(y_array, probability),
            "average_precision": average_precision_score(y_array, probability),
            "log_loss": log_loss(y_array, probability),
            "brier_score": brier_score_loss(y_array, probability),
            "accuracy": accuracy_score(y_array, predictions),
            "balanced_accuracy": balanced_accuracy_score(y_array, predictions),
            "precision": precision_score(y_array, predictions, zero_division=0),
            "recall": recall_score(y_array, predictions, zero_division=0),
            "f1": f1_score(y_array, predictions, zero_division=0),
            "positive_prediction_rate": predictions.mean(),
        }
    )

test_probability = model.predict_proba(X_test_cat)[:, 1]
test_prediction = (test_probability >= selected_threshold).astype("int8")

model_metrics = evaluate_probabilities(y_test, test_probability, selected_threshold)
dummy_metrics = evaluate_probabilities(
    y_test,
    dummy_test_probability,
    threshold=float(y_train.mean()),
)

display(pd.concat({"XGBoost": model_metrics, "Prior dummy": dummy_metrics}, axis=1))

print(classification_report(y_test, test_prediction, digits=4, zero_division=0))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))

ConfusionMatrixDisplay.from_predictions(
    y_test, test_prediction, values_format=",d", ax=axes[0]
)
axes[0].set_title(f"Confusion Matrix\nThreshold={selected_threshold:.4f}")

RocCurveDisplay.from_predictions(
    y_test, test_probability, name="XGBoost", ax=axes[1]
)
axes[1].plot([0, 1], [0, 1], linestyle="--")
axes[1].set_title("ROC Curve")

PrecisionRecallDisplay.from_predictions(
    y_test, test_probability, name="XGBoost", ax=axes[2]
)
axes[2].axhline(y_test.mean(), linestyle="--", label="Click prevalence")
axes[2].set_title("Precision-Recall Curve")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
fraction_positive, mean_predicted = calibration_curve(
    y_test,
    test_probability,
    n_bins=10,
    strategy="quantile",
)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(mean_predicted, fraction_positive, marker="o", label="XGBoost")
ax.plot([0, 1], [0, 1], linestyle="--", label="Perfect calibration")
ax.set_title("Probability Calibration")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Observed click rate")
ax.legend()
plt.tight_layout()
plt.show()

## 15. Campaign targeting versus random at equal send volume

For each send fraction, calculate:

- email savings,
- clicks captured,
- click capture rate,
- targeted CTR,
- CTR lift versus sending to everyone,
- random expected clicks at the same volume,
- incremental clicks versus random,
- click lift over random.

This is a more operational comparison than selecting an arbitrary ROC threshold.

In [ ]:
def targeting_curve(y_true, probability, fractions):
    ranked = pd.DataFrame(
        {"actual": np.asarray(y_true, dtype=int), "probability": probability}
    ).sort_values("probability", ascending=False)

    n_rows = len(ranked)
    total_clicks = int(ranked["actual"].sum())
    overall_rate = ranked["actual"].mean()
    rows = []

    for fraction in fractions:
        target_count = max(1, min(n_rows, int(np.ceil(n_rows * fraction))))
        targeted = ranked.head(target_count)
        targeted_clicks = int(targeted["actual"].sum())
        random_expected_clicks = total_clicks * target_count / n_rows

        rows.append(
            {
                "target_fraction": target_count / n_rows,
                "emails_sent": target_count,
                "email_savings": 1 - target_count / n_rows,
                "clicks_captured": targeted_clicks,
                "click_capture_rate": (
                    targeted_clicks / total_clicks if total_clicks else np.nan
                ),
                "targeted_ctr": targeted["actual"].mean(),
                "ctr_lift_vs_all": (
                    targeted["actual"].mean() / overall_rate if overall_rate else np.nan
                ),
                "random_expected_clicks": random_expected_clicks,
                "incremental_clicks_vs_random": (
                    targeted_clicks - random_expected_clicks
                ),
                "click_lift_vs_random": (
                    targeted_clicks / random_expected_clicks
                    if random_expected_clicks else np.nan
                ),
            }
        )
    return pd.DataFrame(rows)

test_targeting = targeting_curve(y_test, test_probability, TARGET_FRACTIONS)
display(test_targeting)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].plot(
    test_targeting["target_fraction"],
    test_targeting["click_capture_rate"],
    marker="o",
    label="XGBoost targeting",
)
axes[0].plot([0, 1], [0, 1], linestyle="--", label="Random targeting")
axes[0].set_title("Cumulative Click Capture")
axes[0].set_xlabel("Fraction emailed")
axes[0].set_ylabel("Fraction of all clicks captured")
axes[0].legend()

axes[1].plot(
    test_targeting["target_fraction"],
    test_targeting["ctr_lift_vs_all"],
    marker="o",
)
axes[1].axhline(1.0, linestyle="--")
axes[1].set_title("CTR Lift by Send Fraction")
axes[1].set_xlabel("Fraction emailed")
axes[1].set_ylabel("CTR lift versus emailing everyone")

plt.tight_layout()
plt.show()

selected_policy = test_targeting.iloc[
    (test_targeting["target_fraction"] - DEFAULT_TARGET_FRACTION).abs().argmin()
]
print(f"Illustrative top-{selected_policy['target_fraction']:.0%} policy")
display(selected_policy.to_frame("value"))

## 16. Decile lift

D1 contains the 10% of users with the highest predicted click probabilities. A useful model should concentrate clicks in the earliest deciles.

In [ ]:
ranked_test = pd.DataFrame(
    {"actual": y_test.to_numpy(), "probability": test_probability}
).sort_values("probability", ascending=False).reset_index(drop=True)

ranked_test["decile"] = pd.qcut(
    ranked_test.index,
    q=10,
    labels=[f"D{i}" for i in range(1, 11)],
)

decile_lift = (
    ranked_test.groupby("decile", observed=True)
    .agg(
        users=("actual", "size"),
        clicks=("actual", "sum"),
        ctr=("actual", "mean"),
        mean_score=("probability", "mean"),
    )
    .reset_index()
)
decile_lift["lift_vs_overall"] = decile_lift["ctr"] / y_test.mean()
decile_lift["cumulative_click_capture"] = (
    decile_lift["clicks"].cumsum() / decile_lift["clicks"].sum()
)

display(decile_lift)

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.barplot(data=decile_lift, x="decile", y="lift_vs_overall", color="#4C72B0", ax=ax)
ax.axhline(1.0, linestyle="--")
ax.set_title("Click Lift by Score Decile")
ax.set_xlabel("D1 = highest score")
ax.set_ylabel("Lift versus overall test CTR")
plt.tight_layout()
plt.show()

## 17. Model interpretation

Gain importance explains which features improve tree splits. Native SHAP contributions summarize the average magnitude of each feature's contribution to individual raw scores.

These are predictive explanations, not causal effects.

In [ ]:
booster = model.get_booster()

gain_importance = (
    pd.Series(booster.get_score(importance_type="gain"), name="gain")
    .rename_axis("feature")
    .reset_index()
    .sort_values("gain", ascending=False)
)
display(gain_importance)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=gain_importance, x="gain", y="feature", color="#4C72B0", ax=ax)
ax.set_title("XGBoost Feature Importance by Gain")
plt.tight_layout()
plt.show()

In [ ]:
explain_sample = X_test_cat.sample(
    n=min(5_000, len(X_test_cat)),
    random_state=RANDOM_STATE,
)
explain_matrix = xgb.DMatrix(explain_sample, enable_categorical=True)
contributions = booster.predict(explain_matrix, pred_contribs=True)

mean_absolute_contribution = pd.Series(
    np.abs(contributions[:, :-1]).mean(axis=0),
    index=booster.feature_names,
    name="mean_absolute_shap",
).sort_values(ascending=False)

display(mean_absolute_contribution.to_frame())

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(
    x=mean_absolute_contribution.values,
    y=mean_absolute_contribution.index,
    color="#4C72B0",
    ax=ax,
)
ax.set_title("Mean Absolute Native SHAP Contribution")
ax.set_xlabel("Mean absolute contribution to raw score")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

## 18. Volume-aware opportunity tables

`click_gap_vs_overall_rate` compares actual clicks with the number expected if a segment converted at the overall CTR:

\[
\text{gap}=
\text{actual clicks}-
(\text{segment sends}\times\text{overall CTR})
\]

Large negative gaps can identify high-volume areas to investigate, but they do not prove a causal intervention.

In [ ]:
opportunity_tables = {}
for feature in ["email_text", "email_version", "user_country", "weekday"]:
    table = segment_performance(df, feature).sort_values("click_gap_vs_overall_rate")
    opportunity_tables[feature] = table
    print(f"\nOpportunity table: {feature}")
    display(table)

## 19. Predictive versus causal conclusions

### The model can

- rank eligible users by predicted click probability;
- quantify click concentration at different send volumes;
- help allocate a limited campaign budget;
- identify segments for investigation;
- generate hypotheses about content, personalization, timing, geography, and customer history.

### The model cannot prove

Historical recipients were already selected to receive an email. The model predicts **response propensity**, not necessarily the **incremental effect** of sending an email.

A high-scoring user may have clicked or purchased without the email. True incrementality requires a randomized no-email/control group and an uplift or treatment-effect design.

Observed differences between email versions or send times are also not automatically causal unless those choices were randomized.

## 20. A/B test for the targeting policy

A modern experiment compares complete campaign policies:

1. Define the eligible population before scoring.
2. Randomly assign eligible users to:
   - **Control**: current targeting policy.
   - **Treatment**: XGBoost-ranked targeting at the chosen capacity or threshold.
3. Keep content, attribution window, and calendar conditions consistent.
4. Report both:
   - clicks per email sent, and
   - clicks/conversions/revenue per eligible user.
5. Use a two-proportion test, logistic regression, or randomization inference for binary outcomes.
6. Pre-specify the primary metric and guardrails such as unsubscribes, spam complaints, and bounce rate.

An ordinary t-test on individual binary labels is not the preferred analysis.

In [ ]:
def approximate_sample_size_per_group(
    control_rate: float,
    relative_lift: float,
    alpha: float = 0.05,
    power: float = 0.80,
) -> int:
    # Planning approximation for equal-sized groups and a two-sided
    # two-proportion comparison.
    treatment_rate = control_rate * (1 + relative_lift)

    if not 0 < control_rate < 1:
        raise ValueError("control_rate must be between 0 and 1.")
    if not 0 < treatment_rate < 1:
        raise ValueError("The implied treatment rate must be between 0 and 1.")

    pooled_rate = (control_rate + treatment_rate) / 2
    z_alpha = norm.ppf(1 - alpha / 2)
    z_power = norm.ppf(power)

    numerator = (
        z_alpha * np.sqrt(2 * pooled_rate * (1 - pooled_rate))
        + z_power * np.sqrt(
            control_rate * (1 - control_rate)
            + treatment_rate * (1 - treatment_rate)
        )
    ) ** 2
    denominator = (treatment_rate - control_rate) ** 2
    return int(np.ceil(numerator / denominator))

illustrative_relative_lift = 0.20
sample_size = approximate_sample_size_per_group(
    control_rate=ctr,
    relative_lift=illustrative_relative_lift,
)
print(
    f"Illustrative sample per group for a {illustrative_relative_lift:.0%} "
    f"relative CTR lift: {sample_size:,}"
)

## 21. Save model and reproducibility artifacts

The model is saved as JSON because it preserves categorical model information. Metadata records the feature contract, category levels, threshold, test metrics, parameters, and package versions.

In [ ]:
MODEL_PATH = OUTPUT_DIR / "marketing_email_xgboost_model.json"
METADATA_PATH = OUTPUT_DIR / "marketing_email_xgboost_metadata.json"
TEST_PREDICTIONS_PATH = OUTPUT_DIR / "marketing_email_xgboost_test_predictions.csv"
TARGETING_PATH = OUTPUT_DIR / "marketing_email_xgboost_targeting_curve.csv"

model.save_model(MODEL_PATH)

test_output = df.loc[X_test.index].copy()
test_output["click_probability"] = test_probability
test_output["predicted_click"] = test_prediction
test_output["score_rank"] = (
    pd.Series(test_probability, index=test_output.index)
    .rank(method="first", ascending=False)
    .astype(int)
)
test_output.to_csv(TEST_PREDICTIONS_PATH, index=False)
test_targeting.to_csv(TARGETING_PATH, index=False)

metadata = {
    "target": TARGET,
    "prediction_moment": "before email send",
    "features": FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "category_schema": CATEGORY_SCHEMA,
    "selected_threshold": selected_threshold,
    "threshold_reason": threshold_reason,
    "default_target_fraction": DEFAULT_TARGET_FRACTION,
    "best_iteration": int(model.best_iteration),
    "model_parameters": model.get_params(),
    "test_metrics": {key: float(value) for key, value in model_metrics.items()},
    "source_files": {
        "email_table": str(EMAIL_TABLE_PATH),
        "opened_table": str(OPENED_TABLE_PATH),
        "clicked_table": str(CLICKED_TABLE_PATH),
    },
    "versions": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgb.__version__,
    },
}

with METADATA_PATH.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, indent=2, default=str)

for path in [MODEL_PATH, METADATA_PATH, TEST_PREDICTIONS_PATH, TARGETING_PATH]:
    print(path)

## 22. Reusable scoring function

Future candidate data must contain the same six original pre-send fields. The function applies identical cleaning, feature engineering, and category handling.

In [ ]:
REQUIRED_SCORING_COLUMNS = set(CATEGORICAL_FEATURES + BASE_NUMERIC_FEATURES)

def score_campaign_candidates(
    candidates: pd.DataFrame,
    fitted_model: xgb.XGBClassifier = model,
    category_schema = CATEGORY_SCHEMA,
    threshold: float = selected_threshold,
) -> pd.DataFrame:
    missing = sorted(REQUIRED_SCORING_COLUMNS - set(candidates.columns))
    if missing:
        raise ValueError(f"Candidate data is missing columns: {missing}")

    candidate_features = build_model_features(candidates)

    invalid_rows = (
        ~candidate_features["hour"].between(1, 24, inclusive="both")
        | candidate_features["user_past_purchases"].lt(0)
        | candidate_features["hour"].isna()
        | candidate_features["user_past_purchases"].isna()
    )
    if invalid_rows.any():
        raise ValueError(
            f"{int(invalid_rows.sum()):,} rows contain invalid hour or purchase values."
        )

    candidate_features = apply_category_schema(candidate_features, category_schema)
    probabilities = fitted_model.predict_proba(candidate_features)[:, 1]

    result = candidates.copy()
    result["click_probability"] = probabilities
    result["predicted_click"] = (probabilities >= threshold).astype("int8")
    result["priority_rank"] = (
        pd.Series(probabilities)
        .rank(method="first", ascending=False)
        .astype(int)
        .to_numpy()
    )
    return result.sort_values("click_probability", ascending=False)

# Example:
# future_candidates = pd.read_csv("future_campaign_candidates.csv")
# scored_candidates = score_campaign_candidates(future_candidates)
# display(scored_candidates.head(20))

## 23. Recommended next steps

1. Add campaign dates and use future-period validation.
2. Add stable user IDs and prevent repeated users from crossing data splits.
3. Optimize purchases, revenue, margin, or lifetime value—not clicks alone.
4. Add email cost and click/conversion value to select a profitable policy.
5. Maintain a randomized no-email holdout for incrementality measurement.
6. When treatment/control data exists, build an uplift workflow with XGBoost.
7. Monitor score drift, unknown categories, PR-AUC, calibration, lift, revenue, and guardrails.
8. Version every model, feature definition, threshold/policy, and experiment result.

## Reference documentation

- [XGBoost categorical data](https://xgboost.readthedocs.io/en/stable/tutorials/categorical.html)
- [XGBoost scikit-learn estimator and early stopping](https://xgboost.readthedocs.io/en/stable/python/sklearn_estimator.html)
- [XGBoost model IO](https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html)
- [scikit-learn model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)
- [scikit-learn probability calibration](https://scikit-learn.org/stable/modules/calibration.html)